<a href="https://colab.research.google.com/github/Kohei-200/math/blob/main/gradientBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Gradient Boosting

reference: [Ensemble Methods
Foundations and Algorithms.
](https://doi.org/10.1201/9781003587774)

2.5 Gradient Boosting and Implementations

ByZhi-Hua Zhou



---



AdaBoost optimizes exponential loss function and only for classification.

Gradient Boosting allows differentiable loss functions for both regression and classification.

the core differences are
1. **Distribution**: AdaBoost Reweights samples, GB trains on Residuals
2. **Step size**: AdaBoost derives $\alpha_t$ analytically, GB optimizes $\gamma_t$ numerically



---



Training set: $\mathbf{D} = {({X_i, y_i})} \text{ : i = 1 ... m}$

Differentiable loss function: $l(y, H(\mathbf{x}))$

Base learning algo $L$

Training rounds: $T$

**Process**
\begin{align}
&w_{1, i} = 1/m, (i∈{1,...m}) \tag{1}\\
&H_0(\mathbf{x}) = argmin_c\sum_{i=1}^ml(y_i, c) \tag{2}\\
&\text{for t = 1, ....T:}\tag{3}\\
&r_{it} = -∇_{H_{t-1}}l(y_i, H_{t-1}(\mathbf{x})) (i∈{1,...m})\tag{4}\\
&D_t = {(\mathbf{x_i}, r_{it})} (i∈{1,...m})\tag{5}\\
&h_t = L(D_t, \mathbf{w})\tag{6} \\
&\gamma_t = argmin_{\gamma}\sum_{i=1}^ml(y_i, H_{t-1}(\mathbf{x_i}) + \gamma h_t(\mathbf{x_i}))\tag{7}\\
& = argmin_\gamma\sum_{i=1}^m(r_{it} - \gamma h_t(\mathbf{x_i}))^2\tag{7} \\
&H_t(\mathbf{x}) = H_{t-1}(\mathbf{x}) + \eta * \gamma_t h_t(\mathbf{x}))\tag{8}\\
&end \tag{9}\\
&output = H_T(\mathbf{x}) \tag{10} \\
\end{align}


$\epsilon_t$ is now gone and so the distribution reweighting.

Now, instead of it, it measures **residual** ($r_{it}$) to measure how much wrong it was.

The residual is fed by a learner trained to predict the residual.

Thus, instead of reweighting data points, GB directly minimizes the loss.

How did we get? $H_0(\mathbf{x}) = argmin_c\sum_{i=1}^ml(y_i, c)$


---


$H_0$ is a mean of y


---


for squared error loss, $l(y, H_{t-1}\mathbf{x}) = (y - H_{t-1}\mathbf{x})^2$,

$r_{it}$ is $-H_{t-1}(\mathbf{x}) + y$

it is literally a residual, rather than a gradient


---

now, $\gamma$ is a step size. same as $\alpha_t$ in a sense that both are the weights on the learners.

However, $\gamma$ is not in a closed form and is solved numerically via argmin.

GB works with any loss function, so it has to optimise gamma_t directly.

derivative of $(r_{it} - \gamma h_t)^2$ is:

$2(r_{it} - \gamma h_t) * (-h_t)$

$\sum 2(r_{it} - \gamma h_t) * (-h_t) = 0$ means (times -2, and move):

$\sum h_t * r_{it} = \gamma * \sum (h_t)^2$

$\gamma = \frac{\sum_i h_t(\mathbf{x_i})*r_{it}}{\sum_i (h_t(\mathbf{x_i}))^2}$

---

last term is the same as AdaBoost, both build $H$ as a running sum, adding one weighted learner each round.

### **Classification**

In [186]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
import numpy as np

In [187]:
X, y = make_classification(
    n_samples = 500,
    n_features = 2,
    n_redundant = 0,
    random_state = 42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2)

In [188]:
np.mean(y_train)

np.float64(0.4775)

In [189]:
# Initiate H_0
H_0 = np.mean(y_train)
H = H_0
T =  25

h = [None] * T #[i for i in range(T)]
gamma = [None] * T
lr = 0.5
for t in range(T):
  r_it = y_train - H
  D_t = [X_train, r_it]
  h[t] = DecisionTreeRegressor(max_depth=1).fit(D_t[0] ,D_t[1])

  h_pred = h[t].predict(X_train)
  gamma[t] = np.sum(h_pred * r_it) / np.sum(h_pred ** 2)

  #### Introducing **Shrinkage** prevents overfit
  H += lr*gamma[t] * h_pred

lis = [None] * T
for i in range(T):
  lis[i] = lr * gamma[i] * h[i].predict(X_test)
pred = np.sum(lis, axis=0) + H_0

#threshold for classification
pred_class = np.where(pred > 0.5, 1, 0)

In [190]:
np.mean(pred)

np.float64(0.5984391419509613)

In [191]:
acc = np.mean(pred_class == y_test)
acc

np.float64(0.91)

### **Regression**

just take off the threshold

In [192]:
from sklearn.datasets import make_regression

In [193]:
X, y = make_regression(
    n_samples = 500,
    n_features = 2,
    random_state = 42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2)

In [197]:
np.mean(np.square(np.mean(y_test) - y_test))

np.float64(296.5133837961089)

In [194]:
# Initiate H_0
H_0 = np.mean(y_train)
H = H_0
T =  25

h = [None] * T #[i for i in range(T)]
gamma = [None] * T
lr = 0.5
for t in range(T):
  r_it = y_train - H
  D_t = [X_train, r_it]
  h[t] = DecisionTreeRegressor(max_depth=1).fit(D_t[0] ,D_t[1])

  h_pred = h[t].predict(X_train)
  gamma[t] = np.sum(h_pred * r_it) / np.sum(h_pred ** 2)

  #### Introducing **Shrinkage** prevents overfit
  H += lr*gamma[t] * h_pred

lis = [None] * T
for i in range(T):
  lis[i] = lr * gamma[i] * h[i].predict(X_test)
pred = np.sum(lis, axis=0) + H_0

In [196]:
np.mean(np.square(pred - y_test))

np.float64(14.123922310308494)

In [200]:
r2_score = 1-np.mean(np.square(pred - y_test))/np.mean(np.square(np.mean(y_test) - y_test))
r2_score

np.float64(0.9523666617355103)

95% of the variance explained

in contrast to AdaBoost, it can try other loss functions than Squared error which is sensitive to outliers, like **Absolute error**, or **Huber Loss**.